In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hibijibi"

In [3]:
!pip install -q langchain-openai langchain-community langchain-core requests duckduckgo-search langchain-huggingface ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import requests

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [6]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=4d1d8ae207a8c845a52df8a67bf3623e&query={city}'

  response = requests.get(url)

  return response.json()

In [7]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1-0528",
    task="text-generation"
)

llm = ChatHuggingFace(llm=llm)

In [8]:
from langchain_core.prompts import ChatPromptTemplate

react_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a helpful AI agent that uses tools to answer questions.

You MUST follow this format:

Thought: you should think about what to do
Action: the action to take, should be one of the available tools
Action Input: the input to the action
Observation: the result of the action

Repeat Thought/Action/Observation as needed.

When you know the final answer, respond with:

Final Answer: the final answer to the user
"""),
    ("placeholder", "{messages}")
])


In [9]:



from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    prompt=react_prompt
)

In [10]:
response = agent.invoke({
    "messages": [
        ("user", "Find the capital of Madhya Pradesh, then find its current weather condition")
    ]
})

print(response["messages"][-1].content)

<think>
The search for the current weather in Bhopal did not return a clear, immediate result. However, from the snippets, we can see that the current temperature in Bhopal is 16.1°C with Mist conditions. The humidity is 63% and wind is blowing at 7.2 km/h.

Since the tool for getting weather data via API did not work (usage limit reached), we have to rely on the search results. The second search for "current weather in Bhopal" provided some details.

Therefore, the capital of Madhya Pradesh is Bhopal, and its current weather condition is misty with a temperature of 16.1°C.

Final Answer: The capital of Madhya Pradesh is Bhopal. The current weather in Bhopal is misty with a temperature of 16.1°C, humidity at 63%, and wind speed of 7.2 km/h.
</think>

### Final Answer:
The capital of Madhya Pradesh is **Bhopal**.  

As of today, the current weather in Bhopal is **16.1°C** with **Mist** conditions. Humidity is 63%, wind speed is 7.2 km/h, and visibility is 4 km. Sunrise was at 07:04 AM, 